# ai-knot first-run walkthrough

A zero-network notebook you can render on GitHub or run locally. It shows the core `add` → `recall` loop, point-in-time recall, and the path into the browser inspector without requiring any API keys.


In [1]:
from datetime import UTC, datetime
from pathlib import Path

from ai_knot import KnowledgeBase, MemoryType
from ai_knot.storage import create_storage

demo_dir = Path(".ai_knot/notebook-walkthrough")
storage = create_storage("sqlite", base_dir=str(demo_dir))
kb = KnowledgeBase(agent_id="notebook-demo", storage=storage, embed_url="")
kb.clear_all()


## 1. Seed a small memory store

These are intentionally atomic facts so you can see what deterministic recall is doing.


In [2]:
kb.add("User leads a backend team building payment APIs", importance=0.95, tags=["profile"])
kb.add(
    "Preferred language is Python",
    type=MemoryType.PROCEDURAL,
    importance=0.92,
    tags=["language"],
)
kb.add("API framework is FastAPI", importance=0.94, tags=["framework"])
kb.add("Primary database is PostgreSQL", importance=0.94, tags=["database"])
kb.add(
    "Deployments happen on Tuesdays after 15:00 UTC",
    type=MemoryType.PROCEDURAL,
    importance=0.90,
    tags=["runbook"],
)
kb.add(
    "A data migration is planned for 2099",
    event_time=datetime(2099, 1, 1, tzinfo=UTC),
    tags=["future"],
)
print(f"Loaded {len(kb.list_facts())} facts into the demo store.")


Loaded 6 facts into the demo store.\n

## 2. Recall only what the next turn needs

The first helper returns prompt-ready text. The second returns structured `Fact` objects.


In [3]:
query = "what language, framework, and database does the user use?"
print(kb.recall(query, top_k=4))
print()
print([fact.content for fact in kb.recall_facts(query, top_k=4)])


[1] Primary database is PostgreSQL\n[2] API framework is FastAPI\n[3] Preferred language is Python\n[4] User leads a backend team building payment APIs\n\n['Primary database is PostgreSQL', 'API framework is FastAPI', 'Preferred language is Python', 'User leads a backend team building payment APIs']\n

## 3. Point-in-time recall

The future migration is not active in 2026, but becomes visible once the time anchor moves past 2099-01-01.


In [4]:
past = kb.recall_facts("2099 migration", now=datetime(2026, 1, 1, tzinfo=UTC))
future = kb.recall_facts("2099 migration", now=datetime(2099, 1, 2, tzinfo=UTC))
print("2026:", [fact.content for fact in past])
print("2099:", [fact.content for fact in future])


2026: []\n2099: ['A data migration is planned for 2099']\n

## 4. Inspect the store and continue

The notebook keeps everything local. `stats()` shows active facts only, which is why the future migration does not count yet.


In [5]:
stats = kb.stats()
print({"total_facts": stats["total_facts"], "by_type": stats["by_type"]})
print("Next commands:")
print("  ai-knot --storage sqlite --data-dir .ai_knot/notebook-walkthrough serve notebook-demo --port 8000")
print("  python examples/browser_inspector_demo.py")


{'total_facts': 5, 'by_type': {'semantic': 3, 'procedural': 2, 'episodic': 0}}\nNext commands:\n  ai-knot --storage sqlite --data-dir .ai_knot/notebook-walkthrough serve notebook-demo --port 8000\n  python examples/browser_inspector_demo.py\n